# Exercises (Student) - MCP Client with LLM

In [ ]:
!pip install -q mcp nest_asyncio requests

In [ ]:

import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = False  # flip True if GITHUB_TOKEN is set


In [ ]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


In [3]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

@mcp.tool()
def add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    "Multiply two numbers."
    return a * b

@mcp.tool()
def greet(name: str) -> str:
    "Return a greeting string."
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run()

Overwriting server.py


## Exercise 1 (provide answer)

#To-Do: Why is STDIO transport simple for local MCP dev compared to HTTP?

## Exercise 2

In [ ]:
import asyncio
import nest_asyncio
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    # Correction : command = executable seul, args = liste separee (pas "python server.py" dans command)
    params = StdioServerParameters(command=sys.executable, args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()


In [ ]:
# In a new cell
await ex2_connect()
print("Exercise 2: OK (connected and initialized)")


Exercise 2: OK (connected and initialized)


## Exercise 3

In [6]:
!pip install -q mcp nest_asyncio requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.2 MB/s eta 0:00:00


In [ ]:
import asyncio
import nest_asyncio
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex3_list():
    # Correction : "python server.py" n'est pas un executable valide (FileNotFoundError).
    # command doit etre l'executable seul, et "server.py" un argument separe.
    params = StdioServerParameters(command=sys.executable, args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            resources = await session.list_resources()
            print("RESOURCES:", resources)
            tools = await session.list_tools()
            for t in tools.tools:
                print(t.name, t.inputSchema.get("properties", {}))


In [10]:
await ex3_list()

ValidationError: 1 validation error for StdioServerParameters
command
  Input should be a valid string [type=string_type, input_value=['python', 'server.py'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

In [4]:
await ex3_list()

NameError: name 'StdioServerParameters' is not defined

## Exercise 4

#To-Do: Explain how the conversion to llm tool happens in MCP server code ?

In [ ]:

def convert_to_llm_tool(tool):
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like “Add 2 to 20.”

In [ ]:
import asyncio
import json
import os
import re
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

# Correction : stub_plan etait utilise dans call_llm mais jamais defini (NameError).
# Planificateur minimal sans LLM : repere un mot-cle d'outil + des nombres/noms dans le prompt.
def stub_plan(prompt: str, functions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    available = {f["function"]["name"] for f in functions}
    text = prompt.lower()
    numbers = [int(n) for n in re.findall(r"-?\d+", prompt)]

    if "multiply" in text and "multiply" in available and len(numbers) >= 2:
        return [{"name": "multiply", "args": {"a": numbers[0], "b": numbers[1]}}]
    if "add" in text and "add" in available and len(numbers) >= 2:
        return [{"name": "add", "args": {"a": numbers[0], "b": numbers[1]}}]
    if "greet" in text and "greet" in available:
        words = prompt.split()
        name = words[-1].strip("!.,") if words else "there"
        return [{"name": "greet", "args": {"name": name}}]
    return []

def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls


In [ ]:
import sys

async def ex5_run(prompt: str = "Add 2 to 20"):
    params = StdioServerParameters(command=sys.executable, args=["server.py"])
    async with stdio_client(params, errlog=True) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            functions = [convert_to_llm_tool(t) for t in tools.tools]
            calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
            print("tool_calls:", calls)
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                print("result:", [getattr(c, "text", str(c)) for c in result.content])


In [ ]:
await ex5_run("Add 2 to 20")

tool_calls: [{'name': 'add', 'args': {'a': 2, 'b': 20}}]
result: ['22']


## Optional - add multiply(a, b) and rerun